# 📊 Notebook 4: Model Evaluation & Performance Analysis

## CWRU Bearing Fault Detection System

---

### Objectives
1. Load the best trained model and evaluate on the test set
2. Generate a detailed classification report (precision, recall, F1 per class)
3. Visualize confusion matrix with various representations
4. Analyze per-class performance and failure modes
5. Compute safety-critical metrics: False Negative Rate (missed faults)
6. Evaluate binary detection performance (Normal vs Any Fault)
7. Examine prediction confidence distributions
8. t-SNE visualization of learned feature representations

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    precision_recall_fscore_support, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.manifold import TSNE
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader import CWRUDataLoader
from src.data.preprocessing import hybrid_split
from src.models.vibration_cnn import VibrationCNN, count_parameters
from src.training.train import BearingDataset
from src.training.evaluate import ModelEvaluator

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (14, 6), 'figure.dpi': 100})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASS_LABELS = CWRUDataLoader.CLASS_LABELS
CLASS_NAMES = list(CLASS_LABELS.values())
FS = 12000

FAULT_COLORS = {
    0: '#2ecc71',  # Green - Normal
    1: '#e74c3c', 2: '#c0392b', 3: '#a93226',  # Red - Inner Race
    4: '#3498db', 5: '#2980b9', 6: '#21618c',  # Blue - Outer Race  
    7: '#f39c12', 8: '#e67e22', 9: '#d35400',  # Orange - Ball
}

print(f"✅ Environment ready! Device: {DEVICE}")

## 2. Load Model & Prepare Test Data

In [ ]:
# ============================================================
# Load data
# ============================================================
loader = CWRUDataLoader(data_dir=str(PROJECT_ROOT / 'data' / 'cwru'))
signals_dict, metadata_df = loader.load_all_data()
labels_dict = loader.get_labels_dict(metadata_df)

X_train, y_train, X_test, y_test = hybrid_split(
    signals_dict, labels_dict,
    file_train_ratio=0.7, time_train_ratio=0.7,
    window_size=2048, overlap=0.5, fs=FS, seed=42
)

test_dataset = BearingDataset(X_test, y_test, augment=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# ============================================================
# Load best model
# ============================================================
model = VibrationCNN(num_classes=10)

# Try multiple paths
model_paths = [
    PROJECT_ROOT / 'models' / 'best_model.pth',
    PROJECT_ROOT / 'models' / 'final_model.pth',
    PROJECT_ROOT / 'best_model.pth',
]

loaded = False
for path in model_paths:
    if path.exists():
        try:
            checkpoint = torch.load(path, map_location=DEVICE)
            if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
            loaded = True
            print(f"✅ Model loaded from: {path}")
            break
        except:
            continue

if not loaded:
    print("⚠️ No pre-trained model found. Training a fresh model...")
    from src.training.train import train_model
    train_dataset = BearingDataset(X_train, y_train, augment=True)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
    train_model(model, train_loader, test_loader, num_epochs=30, device=DEVICE)
    loaded = True

model.to(DEVICE)
model.eval()
print(f"   Parameters: {count_parameters(model)[0]:,}")

## 3. Generate Predictions

In [ ]:
# ============================================================
# Collect all predictions, labels, and probabilities
# ============================================================
all_preds = []
all_labels = []
all_probs = []
all_features = []

model.eval()
with torch.no_grad():
    for signals, labels in test_loader:
        signals = signals.to(DEVICE)
        
        # Get predictions
        logits = model(signals)
        probs = F.softmax(logits, dim=1)
        _, predicted = logits.max(1)
        
        # Get features (before classifier)
        features = model.extract_features(signals)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_features.extend(features.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
all_features = np.array(all_features)

# Overall accuracy
accuracy = 100.0 * np.mean(all_preds == all_labels)
print(f"\n📊 Test Set Results:")
print(f"  Total test samples: {len(all_labels):,}")
print(f"  Overall Accuracy: {accuracy:.2f}%")
print(f"  Feature shape: {all_features.shape}")

## 4. Classification Report

In [ ]:
# ============================================================
# Detailed classification report
# ============================================================
report = classification_report(
    all_labels, all_preds, 
    target_names=CLASS_NAMES, 
    output_dict=True,
    zero_division=0
)

# Print text report
print("\n" + "="*75)
print("CLASSIFICATION REPORT")
print("="*75)
print(classification_report(all_labels, all_preds, 
                            target_names=CLASS_NAMES, zero_division=0))

# Create a pretty DataFrame
report_df = pd.DataFrame(report).T
report_df = report_df.iloc[:-3]  # Remove macro/weighted avg rows for visualization
report_df = report_df.round(4)

# Visualize per-class metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ['precision', 'recall', 'f1-score']
colors = [FAULT_COLORS[i] for i in range(10)]

for ax, metric in zip(axes, metrics):
    values = [report[name][metric] for name in CLASS_NAMES]
    bars = ax.barh(CLASS_NAMES, values, color=colors, 
                   edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.set_xlim([0, 1.15])
    ax.axvline(x=0.95, color='green', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_title(metric.title(), fontweight='bold', fontsize=13)
    ax.grid(True, alpha=0.3, axis='x')
    
    for bar, val in zip(bars, values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Per-Class Performance Metrics', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Confusion Matrix

In [ ]:
# ============================================================
# Confusion matrices (count and normalized)
# ============================================================
cm = confusion_matrix(all_labels, all_preds)
cm_normalized = confusion_matrix(all_labels, all_preds, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0], square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('True', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold', fontsize=13)
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[1], square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, vmin=0, vmax=1)
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('True', fontsize=12)
axes[1].set_title('Confusion Matrix (Row-Normalized)', fontweight='bold', fontsize=13)
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.suptitle(f'Test Set Confusion Matrix — Overall Accuracy: {accuracy:.2f}%', 
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Error analysis: which classes get confused most?
# ============================================================
print("\n🔍 Misclassification Analysis:")
print("="*60)

errors = all_preds != all_labels
n_errors = np.sum(errors)
print(f"Total misclassifications: {n_errors} / {len(all_labels)} ({100*n_errors/len(all_labels):.2f}%)")

# Find most common confusions
confusion_pairs = []
for true, pred in zip(all_labels[errors], all_preds[errors]):
    confusion_pairs.append((CLASS_NAMES[true], CLASS_NAMES[pred]))

if confusion_pairs:
    from collections import Counter
    common_confusions = Counter(confusion_pairs).most_common(10)
    
    print("\nTop 10 most common misclassifications:")
    for (true_cls, pred_cls), count in common_confusions:
        print(f"  {true_cls:20s} → {pred_cls:20s}: {count:4d} times")
else:
    print("\n🎉 No misclassifications!")

## 6. Safety-Critical Metrics

In fault detection, **missing a fault (False Negative) is worse than a false alarm (False Positive)**.

In [ ]:
# ============================================================
# Binary detection: Normal (0) vs Fault (1-9)
# ============================================================
binary_labels = (all_labels > 0).astype(int)
binary_preds = (all_preds > 0).astype(int)
binary_probs = 1 - all_probs[:, 0]  # P(fault) = 1 - P(normal)

tn = np.sum((binary_labels == 0) & (binary_preds == 0))
fp = np.sum((binary_labels == 0) & (binary_preds == 1))
fn = np.sum((binary_labels == 1) & (binary_preds == 0))
tp = np.sum((binary_labels == 1) & (binary_preds == 1))

fnr = 100.0 * fn / (fn + tp) if (fn + tp) > 0 else 0
fpr = 100.0 * fp / (fp + tn) if (fp + tn) > 0 else 0
detection_accuracy = 100.0 * (tp + tn) / (tp + tn + fp + fn)

print("\n🚨 Binary Fault Detection (Normal vs ANY Fault):")
print("="*60)
print(f"  Detection Accuracy:    {detection_accuracy:.2f}%")
print(f"  True Positive Rate:    {100-fnr:.2f}% (fault detected when present)")
print(f"  True Negative Rate:    {100-fpr:.2f}% (normal detected when normal)")
print(f"  False Negative Rate:   {fnr:.2f}% ⚠️ (faults MISSED)")
print(f"  False Positive Rate:   {fpr:.2f}% (false alarms)")

print(f"\n  Binary Confusion Matrix:")
print(f"                Pred Normal  Pred Fault")
print(f"  True Normal:  {tn:>10d}   {fp:>10d}")
print(f"  True Fault:   {fn:>10d}   {tp:>10d}")

if fnr == 0:
    print(f"\n  ✅ Zero missed faults! Perfect safety performance.")
elif fnr < 1:
    print(f"\n  ✅ FNR < 1% — Excellent safety performance.")
else:
    print(f"\n  ⚠️ FNR = {fnr:.2f}% — {fn} faults were missed. Review model.")

## 7. Prediction Confidence Analysis

In [ ]:
# ============================================================
# Confidence distribution: correct vs incorrect predictions
# ============================================================
max_probs = np.max(all_probs, axis=1)
correct_mask = all_preds == all_labels

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall confidence distribution
axes[0].hist(max_probs, bins=50, color='steelblue', alpha=0.7, 
             edgecolor='black', linewidth=0.5)
axes[0].axvline(np.mean(max_probs), color='red', linestyle='--', 
                label=f'Mean={np.mean(max_probs):.3f}')
axes[0].set_xlabel('Prediction Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Overall Confidence Distribution', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Correct vs incorrect
axes[1].hist(max_probs[correct_mask], bins=50, alpha=0.6, color='green', 
             label=f'Correct (n={np.sum(correct_mask):,})', density=True)
if np.sum(~correct_mask) > 0:
    axes[1].hist(max_probs[~correct_mask], bins=30, alpha=0.6, color='red', 
                 label=f'Incorrect (n={np.sum(~correct_mask):,})', density=True)
axes[1].set_xlabel('Prediction Confidence')
axes[1].set_ylabel('Density')
axes[1].set_title('Confidence: Correct vs Incorrect', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Per-class confidence
class_confidences = []
for cid in range(10):
    mask = (all_labels == cid) & correct_mask
    if np.sum(mask) > 0:
        class_confidences.append(max_probs[mask])
    else:
        class_confidences.append([0])

bp = axes[2].boxplot(class_confidences, patch_artist=True, showfliers=False)
for i, box in enumerate(bp['boxes']):
    box.set_facecolor(FAULT_COLORS[i])
    box.set_alpha(0.7)
axes[2].set_xticklabels([f'C{i}' for i in range(10)], rotation=0)
axes[2].set_xlabel('Class')
axes[2].set_ylabel('Confidence')
axes[2].set_title('Confidence by Class (Correct Only)', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Prediction Confidence Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n📊 Confidence Statistics:")
print(f"  Mean confidence (all):     {np.mean(max_probs):.4f}")
print(f"  Mean confidence (correct): {np.mean(max_probs[correct_mask]):.4f}")
if np.sum(~correct_mask) > 0:
    print(f"  Mean confidence (wrong):   {np.mean(max_probs[~correct_mask]):.4f}")

## 8. ROC Curves (One-vs-Rest)

In [ ]:
# ============================================================
# ROC curves for each class (One-vs-Rest)
# ============================================================
labels_bin = label_binarize(all_labels, classes=range(10))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Individual class ROC curves
auc_scores = {}
for i in range(10):
    if np.sum(labels_bin[:, i]) > 0:
        fpr_i, tpr_i, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
        auc_i = auc(fpr_i, tpr_i)
        auc_scores[i] = auc_i
        axes[0].plot(fpr_i, tpr_i, color=FAULT_COLORS[i], linewidth=1.5,
                     label=f'{CLASS_NAMES[i]} (AUC={auc_i:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.3)
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontsize=11)
axes[0].set_title('ROC Curves (One-vs-Rest)', fontweight='bold', fontsize=13)
axes[0].legend(fontsize=8, loc='lower right')
axes[0].grid(True, alpha=0.3)

# AUC bar chart
auc_values = [auc_scores.get(i, 0) for i in range(10)]
colors_bar = [FAULT_COLORS[i] for i in range(10)]
bars = axes[1].barh(CLASS_NAMES, auc_values, color=colors_bar, 
                    edgecolor='black', linewidth=0.5, alpha=0.85)
axes[1].set_xlim([0.9, 1.01])
axes[1].axvline(x=0.99, color='green', linestyle='--', alpha=0.5)
axes[1].set_xlabel('AUC Score')
axes[1].set_title('Per-Class AUC Scores', fontweight='bold', fontsize=13)
axes[1].grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, auc_values):
    axes[1].text(val + 0.001, bar.get_y() + bar.get_height()/2, 
                 f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

mean_auc = np.mean(auc_values)
print(f"\n📊 Mean AUC (macro): {mean_auc:.4f}")

## 9. t-SNE Feature Visualization

Visualize the learned 128-dimensional features in 2D to see how well the model separates fault types.

In [ ]:
# ============================================================
# t-SNE visualization of learned features
# ============================================================
print("🔄 Running t-SNE on learned features (this may take a minute)...")

# Subsample if dataset is large
max_samples = 2000
if len(all_features) > max_samples:
    indices = np.random.choice(len(all_features), max_samples, replace=False)
    features_sub = all_features[indices]
    labels_sub = all_labels[indices]
    preds_sub = all_preds[indices]
else:
    features_sub = all_features
    labels_sub = all_labels
    preds_sub = all_preds

# Run t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, 
            n_iter=1000, learning_rate='auto', init='pca')
features_2d = tsne.fit_transform(features_sub)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# By true label
for cid in range(10):
    mask = labels_sub == cid
    axes[0].scatter(features_2d[mask, 0], features_2d[mask, 1], 
                    c=FAULT_COLORS[cid], label=CLASS_NAMES[cid],
                    s=15, alpha=0.6, edgecolors='none')
axes[0].set_title('t-SNE of Learned Features (True Labels)', fontweight='bold', fontsize=13)
axes[0].legend(fontsize=8, markerscale=2, loc='best')
axes[0].grid(True, alpha=0.2)
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')

# Highlight misclassifications
correct = preds_sub == labels_sub
axes[1].scatter(features_2d[correct, 0], features_2d[correct, 1], 
                c='green', s=10, alpha=0.3, label=f'Correct ({np.sum(correct)})', edgecolors='none')
if np.sum(~correct) > 0:
    axes[1].scatter(features_2d[~correct, 0], features_2d[~correct, 1], 
                    c='red', s=40, alpha=0.8, label=f'Misclassified ({np.sum(~correct)})',
                    edgecolors='black', linewidths=0.5, marker='X')
axes[1].set_title('t-SNE — Correct vs Misclassified', fontweight='bold', fontsize=13)
axes[1].legend(fontsize=10, markerscale=2)
axes[1].grid(True, alpha=0.2)
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

plt.tight_layout()
plt.show()

print("\n✅ Well-separated clusters indicate the model has learned discriminative features")
print("   Misclassified points at cluster boundaries are expected")

## 10. Performance Summary

In [ ]:
# ============================================================
# Final summary dashboard
# ============================================================
print("\n" + "="*70)
print("📋 FINAL EVALUATION SUMMARY")
print("="*70)

precision, recall, f1, support = precision_recall_fscore_support(
    all_labels, all_preds, average='weighted', zero_division=0
)

summary_data = {
    'Metric': [
        'Overall Accuracy', 'Weighted Precision', 'Weighted Recall', 
        'Weighted F1-Score', 'Mean AUC (macro)', 
        'False Negative Rate', 'False Positive Rate',
        'Mean Confidence (correct)', 'Test Samples'
    ],
    'Value': [
        f'{accuracy:.2f}%', f'{precision*100:.2f}%', f'{recall*100:.2f}%',
        f'{f1*100:.2f}%', f'{mean_auc:.4f}',
        f'{fnr:.2f}%', f'{fpr:.2f}%',
        f'{np.mean(max_probs[correct_mask]):.4f}', f'{len(all_labels):,}'
    ],
    'Target': [
        '> 95%', '> 95%', '> 95%', '> 95%', '> 0.99',
        '< 1%', '< 5%', '> 0.90', '-'
    ],
    'Status': [
        '✅' if accuracy > 95 else '⚠️',
        '✅' if precision > 0.95 else '⚠️',
        '✅' if recall > 0.95 else '⚠️',
        '✅' if f1 > 0.95 else '⚠️',
        '✅' if mean_auc > 0.99 else '⚠️',
        '✅' if fnr < 1 else '⚠️',
        '✅' if fpr < 5 else '⚠️',
        '✅' if np.mean(max_probs[correct_mask]) > 0.90 else '⚠️',
        '✅'
    ]
}

summary_df = pd.DataFrame(summary_data)
display(summary_df)

targets_met = sum(1 for s in summary_data['Status'] if s == '✅')
total_targets = len(summary_data['Status'])
print(f"\n🎯 Targets met: {targets_met}/{total_targets}")